In [7]:
# ============================================================
# CNN-GRU + ATTENTION
# FULL TUNING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    GRU,
    Dense,
    Dropout,
    MaxPooling1D,
    Attention,
    Concatenate,
    GlobalAveragePooling1D
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


# ============================================================
# METRICS
# ============================================================

def nse(y_true, y_pred):

    denom = np.sum((y_true - np.mean(y_true))**2)

    if denom == 0:
        return 0

    return 1 - np.sum((y_true - y_pred)**2) / denom


def compute_metrics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    nse_val = nse(y_true, y_pred)

    return rmse, mae, nse_val


# ============================================================
# PREPARE TIME SERIES
# ============================================================

def prepare_time_series(df, value_name='flow'):

    df.columns = df.columns.str.strip().str.lower()

    months = [
        "jan","feb","mar","apr","maj","jun",
        "jul","avg","sep","okt","nov","dec"
    ]

    existing_months = [m for m in months if m in df.columns]

    # numeric
    df[existing_months] = df[existing_months].apply(
        pd.to_numeric,
        errors='coerce'
    )

    # fill missing
    df[existing_months] = (
        df[existing_months]
        .fillna(method='ffill')
        .fillna(method='bfill')
    )

    # long format
    ts = df.melt(
        id_vars='year',
        value_vars=existing_months,
        var_name='month',
        value_name=value_name
    )

    ts['date'] = pd.to_datetime(
        ts['year'].astype(str) + '-' + ts['month'],
        format='%Y-%b',
        errors='coerce'
    )

    ts = ts.dropna(subset=['date'])

    ts = ts.set_index('date').sort_index()

    ts = ts.dropna(subset=[value_name])

    ts[value_name] = ts[value_name].clip(lower=0)

    return ts


# ============================================================
# PREPARE ML DATA
# ============================================================

def prepare_ml_data(
    ts,
    window_size=24,
    split_date='2014-01-01'
):

    ts = ts.copy()

    ts['flow'] = ts['flow'].clip(lower=0)

    # seasonality
    ts['sin_month'] = np.sin(
        2 * np.pi * ts.index.month / 12
    )

    ts['cos_month'] = np.cos(
        2 * np.pi * ts.index.month / 12
    )

    flows = ts['flow'].values
    sin_m = ts['sin_month'].values
    cos_m = ts['cos_month'].values
    dates = ts.index

    X, y, idx = [], [], []

    for i in range(window_size, len(flows)):

        lag_vals = flows[i-window_size:i]

        features = np.concatenate([
            lag_vals,
            [sin_m[i], cos_m[i]]
        ])

        X.append(features)
        y.append(flows[i])
        idx.append(dates[i])

    X = np.array(X)
    y = np.array(y)

    idx = pd.to_datetime(idx)

    # log-transform
    y_log = np.log1p(y)

    split_date = pd.to_datetime(split_date)

    split_idx = np.where(idx < split_date)[0]

    split = split_idx[-1] + 1

    return {

        "X_train": X[:split],
        "y_train": y_log[:split],

        "X_test": X[split:],
        "y_test": y[split:],

        "idx_test": idx[split:]
    }


# ============================================================
# BUILD CNN-GRU-ATTENTION MODEL
# ============================================================
def build_gru_attention(input_shape):

    inputs = Input(shape=input_shape)

    # GRU sequence
    x = GRU(64, return_sequences=True)(inputs)

    # Self-attention
    attention = Attention()([x, x])

    # Combine GRU + attention
    x = Concatenate()([x, attention])

    # Pooling
    x = GlobalAveragePooling1D()(x)

    # Output
    outputs = Dense(1)(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


# ============================================================
# TUNE CNN-GRU-ATTENTION
# ============================================================

def tune_cnn_gru_attention(X_train, y_train):

    param_grid = {

        'filters': [16, 32],

        'gru_units': [32, 64],

        'dropout_rate': [0.2, 0.3],

        'learning_rate': [0.001, 0.0005],

        'batch_size': [32, 64, 128]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    best_score = -np.inf
    best_params = None

    for filters, gru_units, dropout_rate, lr, batch_size in itertools.product(

        param_grid['filters'],
        param_grid['gru_units'],
        param_grid['dropout_rate'],
        param_grid['learning_rate'],
        param_grid['batch_size']
    ):

        scores = []

        for train_idx, val_idx in tscv.split(X_train):

            X_tr, X_val = X_train[train_idx], X_train[val_idx]

            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            # scaling
            scaler = StandardScaler()

            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

            # reshape
            X_tr = X_tr.reshape(
                (X_tr.shape[0], X_tr.shape[1], 1)
            )

            X_val = X_val.reshape(
                (X_val.shape[0], X_val.shape[1], 1)
            )

            model = build_cnn_gru_attention(

                input_shape=(X_tr.shape[1], 1),

                filters=filters,

                gru_units=gru_units,

                dropout_rate=dropout_rate,

                learning_rate=lr
            )

            early_stop = EarlyStopping(

                monitor='val_loss',

                patience=15,

                restore_best_weights=True
            )

            model.fit(

                X_tr,
                y_tr,

                validation_data=(X_val, y_val),

                epochs=200,

                batch_size=batch_size,

                verbose=0,

                callbacks=[early_stop]
            )

            # prediction
            y_pred_log = model.predict(
                X_val,
                verbose=0
            ).flatten()

            y_pred = np.expm1(y_pred_log)

            y_pred = np.maximum(y_pred, 0)

            y_true = np.expm1(y_val)

            score = nse(y_true, y_pred)

            scores.append(score)

        avg_score = np.mean(scores)

        print(
            filters,
            gru_units,
            dropout_rate,
            lr,
            batch_size,
            "NSE:",
            round(avg_score, 4)
        )

        if avg_score > best_score:

            best_score = avg_score

            best_params = (
                filters,
                gru_units,
                dropout_rate,
                lr,
                batch_size
            )

    print("\n===================================")
    print("BEST CNN-GRU-ATTENTION CONFIG")
    print("===================================")

    print("Best NSE:", best_score)

    print("Params:", best_params)

    # ========================================================
    # FINAL TRAINING ON FULL TRAIN DATA
    # ========================================================

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X_train)

    X_scaled = X_scaled.reshape(
        (X_scaled.shape[0], X_scaled.shape[1], 1)
    )

    final_model = build_cnn_gru_attention(

        input_shape=(X_scaled.shape[1], 1),

        filters=best_params[0],

        gru_units=best_params[1],

        dropout_rate=best_params[2],

        learning_rate=best_params[3]
    )

    final_model.fit(

        X_scaled,
        y_train,

        epochs=100,

        batch_size=best_params[4],

        verbose=0
    )

    return final_model, scaler




In [8]:
# ============================================================
# TEST MODEL
# ============================================================

def test_cnn_gru_attention(model, scaler, data):

    X_test = data["X_test"]

    y_test = data["y_test"]

    idx = data["idx_test"]

    # scale
    X_test = scaler.transform(X_test)

    # reshape
    X_test = X_test.reshape(
        (X_test.shape[0], X_test.shape[1], 1)
    )

    # predict
    y_pred_log = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = np.expm1(y_pred_log)

    y_pred = np.maximum(y_pred, 0)

    rmse, mae, nse_val = compute_metrics(
        y_test,
        y_pred
    )

    print("\n==============================")
    print("CNN-GRU-ATTENTION PERFORMANCE")
    print("==============================")

    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"NSE : {nse_val:.4f}")

    # plot
    plt.figure(figsize=(14,5))

    plt.plot(idx, y_test, label='Actual')

    plt.plot(
        idx,
        y_pred,
        label='CNN-GRU-Attention',
        linestyle='--'
    )

    plt.title(
        "CNN-GRU-Attention Prediction (2014–2025)"
    )

    plt.xlabel("Date")

    plt.ylabel("Flow")

    plt.legend()

    plt.grid(True)

    plt.show()

    return y_pred


In [ ]:

# ============================================================
# EXECUTE WORKFLOW
# ============================================================

q_min = pd.read_csv("q_min.csv")

q_min.columns = q_min.columns.str.strip().str.lower()

ts_qmin = prepare_time_series(q_min)

data = prepare_ml_data(
    ts_qmin,
    window_size=24
)

X_train = data["X_train"]
y_train = data["y_train"]

model, scaler = tune_cnn_gru_attention(
    X_train,
    y_train
)

y_pred = test_cnn_gru_attention(
    model,
    scaler,
    data
)

16 32 0.2 0.001 32 NSE: 0.123
